# **StarSight**
## *AI-Enabled Detection of Exoplanets from Noisy Astronomical Light Curves*

### **Milestone 4: AstroNet Feature Generation**

**Objective:**
Engineer uniform matrix inputs compatible with AstroNet-like Convolutional Neural Networks (CNNs). We will bin phase-folded light curves into a 2001-bin Global View (representing the entire orbital phase range) and a 201-bin Local View (representing a high-resolution window centered on the transit dip). Additionally, we will extract stellar parameters ($T_{eff}$, $R_{*}$, $\log g$) directly from Kepler telemetry FITS headers and serialize the unified dataset.

### **1. Environment Setup**
Mount Google Drive if executing in Google Colab, import required libraries, and configure paths dynamically.

In [1]:
import sys
from pathlib import Path

# Detect if running in Google Colab vs. local workspace
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/StarSight')
    # Install dependencies silently
    !pip install -q lightkurve astropy tqdm pandas numpy matplotlib scipy
else:
    print("Running in local environment.")
    notebook_dir = Path.cwd()
    BASE_DIR = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir

print(f"StarSight project root directory resolved to: {BASE_DIR.resolve()}")

Running in local environment.
StarSight project root directory resolved to: /Users/suryanshdixit/Desktop/StarSight


### **2. Imports**
Import scientific computing libraries, interpolation tools, and data trackers.

In [2]:
import logging
import socket
from typing import Tuple, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from tqdm import tqdm

socket.setdefaulttimeout(30)

### **3. Configuration & Dimensionality Parameters**
Configure absolute dimensions for binned arrays, window size multipliers, and directory structures.

In [3]:
# AstroNet Dimensionality Parameters
GLOBAL_BINS = 2001              # Fixed size of the full phase curve vector
LOCAL_BINS = 201                # Fixed size of the zoomed transit window vector
LOCAL_WINDOW_MULTIPLIER = 4.0   # Window size centered around 0.0 (equal to 4x transit duration)

# Path Configurations
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RESULTS_DIR = BASE_DIR / "results"
PLOTS_DIR = BASE_DIR / "results" / "feature_plots"

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] StarSight.FeatureGen - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger("StarSight.FeatureGen")

### **4. Modular AstroNet Binning Functions**
Define core functions for transit parameter loading, global/local binning, and stellar parameters extraction.

In [4]:
def load_transit_parameters(csv_path: Path) -> pd.DataFrame:
    """
    Load periodic transit parameters from the summary tracking CSV.
    
    Args:
        csv_path: Path to the transit summary CSV file.
        
    Returns:
        pd.DataFrame: Table of transit parameters for each target.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"Transit summary file not found: {csv_path}")
    return pd.read_csv(csv_path)

def load_processed_data(file_path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load clean time and flux arrays from the preprocessed target npz file.
    
    Args:
        file_path: Absolute path to the .npz archive.
        
    Returns:
        Tuple[np.ndarray, np.ndarray]: time and flux arrays.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"Processed file not found: {file_path}")
    data = np.load(file_path)
    return data['time'], data['flux']

def generate_global_view(time_folded: np.ndarray, flux_folded: np.ndarray, num_bins: int) -> np.ndarray:
    """
    Uniformly bin the full phase range [-0.5, 0.5] into a fixed-size global view array.
    Applies linear interpolation to fill empty bins, and centers the median at exactly 0.0.
    
    Args:
        time_folded: 1D phase array of folded time values (typically centered in [-0.5, 0.5]).
        flux_folded: 1D normalized flux array.
        num_bins: Target size of the output binned array.
        
    Returns:
        np.ndarray: Binned and normalized global view vector of size num_bins.
    """
    bin_edges = np.linspace(-0.5, 0.5, num_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    
    bin_indices = np.digitize(time_folded, bin_edges) - 1
    
    binned_flux = np.zeros(num_bins)
    for i in range(num_bins):
        mask = (bin_indices == i)
        if np.any(mask):
            binned_flux[i] = np.median(flux_folded[mask])
        else:
            binned_flux[i] = np.nan
            
    # Fill empty bins using linear interpolation
    nan_mask = np.isnan(binned_flux)
    if np.any(nan_mask):
        if np.all(nan_mask):
            binned_flux = np.zeros(num_bins)
        else:
            x_nonan = bin_centers[~nan_mask]
            y_nonan = binned_flux[~nan_mask]
            f_interp = interp1d(x_nonan, y_nonan, kind='linear', fill_value='extrapolate')
            binned_flux[nan_mask] = f_interp(bin_centers[nan_mask])
            
    # Shift median to exactly 0.0
    binned_flux -= np.median(binned_flux)
    return binned_flux

def generate_local_view(time_folded: np.ndarray, flux_folded: np.ndarray, duration: float, period: float, num_bins: int) -> np.ndarray:
    """
    Uniformly bin the high-resolution zoomed transit window into a fixed-size local view array.
    Filters the time range within [-2 * duration/period, 2 * duration/period].
    
    Args:
        time_folded: 1D phase array of folded time values.
        flux_folded: 1D normalized flux array.
        duration: Transit duration in days.
        period: Orbital period in days.
        num_bins: Target size of the output binned array.
        
    Returns:
        np.ndarray: Binned and normalized local view vector of size num_bins.
    """
    # Zoom window boundary calculation: 4x the transit duration (centered at Phase 0.0)
    half_width = 2.0 * duration / period
    half_width = min(half_width, 0.5) # Cap window size to phase limits
    
    bin_edges = np.linspace(-half_width, half_width, num_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    
    bin_indices = np.digitize(time_folded, bin_edges) - 1
    
    binned_flux = np.zeros(num_bins)
    for i in range(num_bins):
        mask = (bin_indices == i)
        if np.any(mask):
            binned_flux[i] = np.median(flux_folded[mask])
        else:
            binned_flux[i] = np.nan
            
    # Fill empty bins using local linear interpolation or fallback to global interpolation
    nan_mask = np.isnan(binned_flux)
    if np.any(nan_mask):
        if np.all(nan_mask):
            # Fallback: if all local bins are empty, interpolate from the full folded dataset
            f_global = interp1d(time_folded, flux_folded, kind='linear', fill_value='extrapolate')
            binned_flux = f_global(bin_centers)
        else:
            x_nonan = bin_centers[~nan_mask]
            y_nonan = binned_flux[~nan_mask]
            f_interp = interp1d(x_nonan, y_nonan, kind='linear', fill_value='extrapolate')
            binned_flux[nan_mask] = f_interp(bin_centers[nan_mask])
            
    # Shift median to exactly 0.0
    binned_flux -= np.median(binned_flux)
    return binned_flux

def compile_stellar_metadata(target_id: str, raw_dir: Path) -> np.ndarray:
    """
    Extract Teff, Stellar Radius, and Surface Gravity (log g) from raw FITS primary header.
    Falls back to standard Sun-like values if parameters cannot be read.
    
    Args:
        target_id: Target star identifier.
        raw_dir: Path to raw FITS files directory.
        
    Returns:
        np.ndarray: 3-element float32 vector containing [Teff, Radius, logg].
    """
    target_clean = target_id.replace(' ', '-').lower()
    fits_files = list(raw_dir.glob(f"{target_clean}_*.fits"))
    
    teff, radius, logg = 5778.0, 1.0, 4.43 # Default Solar parameters
    
    if fits_files:
        try:
            from astropy.io import fits
            with fits.open(fits_files[0]) as hdul:
                hdr = hdul[0].header
                teff = float(hdr.get("TEFF", teff))
                radius = float(hdr.get("RADIUS", radius))
                logg = float(hdr.get("LOGG", logg))
                logger.info(f"Loaded stellar parameters for {target_id}: Teff={teff}K, R={radius}R_sun, logg={logg}")
        except Exception as e:
            logger.warning(f"Error reading FITS header for target {target_id}: {str(e)}")
    else:
        logger.warning(f"No FITS files found for {target_id} in raw directory. Using Solar fallbacks.")
        
    return np.array([teff, radius, logg], dtype=np.float32)

### **5. Diagnostic Plotting**
Define subplot visualizations for global views vs. zoomed-in local views.

In [5]:
def plot_astronet_features(target_id: str, global_view: np.ndarray, local_view: np.ndarray, save_path: Path) -> None:
    """
    Generate a side-by-side diagnostic panel showing the 2001-bin global view 
    and the 201-bin local zoom view.
    
    Args:
        target_id: Target star identifier.
        global_view: 2001-element global view array.
        local_view: 201-element local view array.
        save_path: Directory path to save the generated png.
    """
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 6))
    
    # Left Panel: Global View
    axes[0].plot(np.linspace(-0.5, 0.5, len(global_view)), global_view, color='black', linewidth=0.8)
    axes[0].set_title(f"AstroNet Global View (2001 Bins): {target_id}", fontsize=11, fontweight='bold')
    axes[0].set_xlabel("Phase", fontsize=10)
    axes[0].set_ylabel("Normalized Median-Subtracted Flux", fontsize=10)
    axes[0].grid(True, linestyle='--', alpha=0.5)
    
    # Right Panel: Local Zoom
    axes[1].plot(np.linspace(-0.5, 0.5, len(local_view)), local_view, color='blue', linewidth=1.2)
    axes[1].set_title(f"AstroNet Local View (201 Bins): {target_id}", fontsize=11, fontweight='bold')
    axes[1].set_xlabel("Zoomed Phase Window", fontsize=10)
    axes[1].set_ylabel("Normalized Flux", fontsize=10)
    axes[1].grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    logger.info(f"Saved AstroNet feature plots to: {save_path.resolve()}")

### **6. Main AstroNet Ingestion and Matrix Stacking Loop**
Load data folders, compute global/local binned arrays, plot features, stack matrices, and serialize dataset.

In [6]:
# Create directories
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load central metrics catalog CSV
csv_path = RESULTS_DIR / "transit_summary.csv"
df_transits = load_transit_parameters(csv_path)

global_features_list = []
local_features_list = []
stellar_features_list = []
binary_labels_list = []

logger.info(f"Found {len(df_transits)} transit targets to engineer. Commencing pipeline...")

for idx, row in df_transits.iterrows():
    raw_target_name = row["Target"]  # e.g., Kepler 10
    # Clean file reference name
    target_clean = raw_target_name.replace(' ', '_').lower()
    npz_filename = f"{target_clean}_processed.npz"
    npz_filepath = PROCESSED_DIR / npz_filename
    
    logger.info(f"Engineering features for target: {raw_target_name}")
    
    # 1. Load preprocessed time and flux
    try:
        time, flux = load_processed_data(npz_filepath)
    except Exception as e:
        logger.error(f"Error loading processed file {npz_filename}: {str(e)}")
        continue
        
    period = float(row["Discovered Period (days)"])
    t0 = float(row["Transit Epoch (t0)"])
    duration = float(row["Duration (days)"])
    
    # 2. Phase-fold time arrays (Time to Phase centering around 0.0 in range [-0.5, 0.5])
    phase = ((time - t0) / period) % 1.0
    phase_centered = np.where(phase > 0.5, phase - 1.0, phase)
    sort_idx = np.argsort(phase_centered)
    phase_sorted = phase_centered[sort_idx]
    flux_sorted = flux[sort_idx]
    
    # 3. Generate Global View (2001 bins)
    global_view = generate_global_view(phase_sorted, flux_sorted, num_bins=GLOBAL_BINS)
    
    # 4. Generate Local View (201 bins)
    local_view = generate_local_view(phase_sorted, flux_sorted, duration, period, num_bins=LOCAL_BINS)
    
    # 5. Compile Stellar features
    stellar_vec = compile_stellar_metadata(raw_target_name, RAW_DIR)
    
    # 6. Append to list trackers
    global_features_list.append(global_view)
    local_features_list.append(local_view)
    stellar_features_list.append(stellar_vec)
    # Labeled as 1 for confirmed exoplanet target
    binary_labels_list.append(1)
    
    # 7. Generate diagnostic plots
    plot_filename = f"{target_clean}_features.png"
    plot_filepath = PLOTS_DIR / plot_filename
    plot_astronet_features(raw_target_name, global_view, local_view, plot_filepath)

# 8. Stack outputs into unified matrix matrices
X_global = np.stack(global_features_list, axis=0)
X_local = np.stack(local_features_list, axis=0)
X_stellar = np.stack(stellar_features_list, axis=0)
y = np.array(binary_labels_list, dtype=np.int32)

# 9. Save consolidated dataset
final_npz_path = PROCESSED_DIR / "final_dataset.npz"
np.savez_compressed(
    final_npz_path,
    X_global=X_global,
    X_local=X_local,
    X_stellar=X_stellar,
    y=y
)

logger.info(f"AstroNet Feature Generation Loop Complete.")
logger.info(f"Consolidated dataset saved to: {final_npz_path.resolve()}")

2026-06-27 03:31:21,234 [INFO] StarSight.FeatureGen - Found 8 transit targets to engineer. Commencing pipeline...


2026-06-27 03:31:21,236 [INFO] StarSight.FeatureGen - Engineering features for target: Kepler 10


2026-06-27 03:31:21,413 [INFO] StarSight.FeatureGen - Loaded stellar parameters for Kepler 10: Teff=5627.0K, R=1.056R_sun, logg=4.342


2026-06-27 03:31:21,616 [INFO] StarSight.FeatureGen - Saved AstroNet feature plots to: /Users/suryanshdixit/Desktop/StarSight/results/feature_plots/kepler_10_features.png


2026-06-27 03:31:21,616 [INFO] StarSight.FeatureGen - Engineering features for target: Kepler 186


...

2026-06-27 03:31:22,723 [INFO] StarSight.FeatureGen - Engineering features for target: Kepler 90


2026-06-27 03:31:22,747 [INFO] StarSight.FeatureGen - Loaded stellar parameters for Kepler 90: Teff=5970.0K, R=1.2R_sun, logg=4.317


2026-06-27 03:31:22,917 [INFO] StarSight.FeatureGen - Saved AstroNet feature plots to: /Users/suryanshdixit/Desktop/StarSight/results/feature_plots/kepler_90_features.png


2026-06-27 03:31:22,927 [INFO] StarSight.FeatureGen - AstroNet Feature Generation Loop Complete.


2026-06-27 03:31:22,927 [INFO] StarSight.FeatureGen - Consolidated dataset saved to: /Users/suryanshdixit/Desktop/StarSight/data/processed/final_dataset.npz


### **7. AstroNet Dataset Dimensions Report**
Confirm dataset dimensions match absolute shapes required by standard CNN input layers.

In [7]:
print("="*60)
print("             STAR SIGHT ASTRONET DATASET SUMMARY")
print("="*60)
print(f"Global Feature Matrix (X_global) shape : {X_global.shape}")
print(f"Local Feature Matrix (X_local) shape   : {X_local.shape}")
print(f"Stellar Feature Matrix (X_stellar) shape : {X_stellar.shape}")
print(f"Label Vector (y) shape                 : {y.shape}")
print(f"Dataset File Path                      : {final_npz_path.resolve()}")
print("="*60)

             STAR SIGHT ASTRONET DATASET SUMMARY
Global Feature Matrix (X_global) shape : (8, 2001)
Local Feature Matrix (X_local) shape   : (8, 201)
Stellar Feature Matrix (X_stellar) shape : (8, 3)
Label Vector (y) shape                 : (8,)
Dataset File Path                      : /Users/suryanshdixit/Desktop/StarSight/data/processed/final_dataset.npz


### **8. Milestone 4 Insights & Next Steps**

### Feature Engineering Findings
- Constructed uniform shape vectors matching input channels of exoplanet classifier models.
- Successfully resolved the phase-folding coordinates to place transit profiles at Phase 0.0.
- Combined high-dimensional light curve inputs with stellar physical properties (temperature, radius, gravity) for late-stage fusion.
- Verified serialized shapes ($8 \times 2001$ for global views, $8 \times 201$ for local views, and $8 \times 3$ for stellar vectors).

### Next Steps
- The next milestone in the StarSight pipeline is **Milestone 5: Convolutional Neural Network (CNN) Classifier Setup** (building the PyTorch/TensorFlow network layers, compiling data loaders, and performing inference).